In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

## 1. Load Dataset

In [2]:
high = pd.read_csv('high_popularity_spotify_data.csv')
low = pd.read_csv('low_popularity_spotify_data.csv')
df = pd.concat([high, low], ignore_index=True)
print('Total songs:', df.shape)
df.head()

Total songs: (4831, 29)


,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,instrumentalness,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3.0,0.0304,...,0.0000,10FLjwfpbxLmW8c25Xyc2N,0.0,6.0,251668.0,0.3080,2plbrEY59IikOBgBGLjaoe,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,4.0,0.0358,...,0.0608,7aJuG4TFXa2hmE4z1yxc3n,1.0,2.0,210373.0,0.2000,6dOtVTDdiauQNBQEDOtlAB,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,4.0,0.0368,...,0.0000,0hBRqPYPXhr1RkTDG3n4Mk,1.0,1.0,166300.0,0.2140,7ne4VBA60CxGM75vw0EYad,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
3,0.910,112.966,0.670,pop,-4.070,0.304,0.786,Sabrina Carpenter,4.0,0.0634,...,0.0000,4B4Elma4nNDUyl6D5PvQkj,0.0,0.0,157280.0,0.0939,1d7Ptw3qYcfpdLNL5REhtJ,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
4,0.783,149.027,0.777,pop,-4.477,0.355,0.939,"ROSÉ, Bruno Mars",4.0,0.2600,...,0.0000,2IYQwwgxgOIn7t3iF6ufFD,0.0,0.0,169917.0,0.0283,5vNRhkKd0yEAg8suGBpjeY,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M


In [3]:
df.columns

Index(['energy', 'tempo', 'danceability', 'playlist_genre', 'loudness',
       'liveness', 'valence', 'track_artist', 'time_signature', 'speechiness',
       'track_popularity', 'track_href', 'uri', 'track_album_name',
       'playlist_name', 'analysis_url', 'track_id', 'track_name',
       'track_album_release_date', 'instrumentalness', 'track_album_id',
       'mode', 'key', 'duration_ms', 'acousticness', 'id', 'playlist_subgenre',
       'type', 'playlist_id'],
      dtype='object')

In [4]:
df.isnull().sum()

energy                      1
tempo                       1
danceability                1
playlist_genre              0
loudness                    1
liveness                    1
valence                     1
track_artist                0
time_signature              1
speechiness                 1
track_popularity            0
track_href                  1
uri                         1
track_album_name            1
playlist_name               0
analysis_url                1
track_id                    0
track_name                  0
track_album_release_date    0
instrumentalness            1
track_album_id              0
mode                        1
key                         1
duration_ms                 1
acousticness                1
id                          1
playlist_subgenre           0
type                        1
playlist_id                 0
dtype: int64

## 2. Data Preprocessing

In [5]:
music = df[['track_id', 'track_name', 'track_artist', 'playlist_genre', 'playlist_subgenre']].copy()
music.head()

,track_id,track_name,track_artist,playlist_genre,playlist_subgenre
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",pop,mainstream
1,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,pop,mainstream
2,7ne4VBA60CxGM75vw0EYad,That’s So True,Gracie Abrams,pop,mainstream
3,1d7Ptw3qYcfpdLNL5REhtJ,Taste,Sabrina Carpenter,pop,mainstream
4,5vNRhkKd0yEAg8suGBpjeY,APT.,"ROSÉ, Bruno Mars",pop,mainstream


In [6]:
music.drop_duplicates(subset=['track_name', 'track_artist'], inplace=True)
music.reset_index(drop=True, inplace=True)
print('After dedup:', music.shape)

After dedup: (4466, 5)


In [7]:
music.dropna(inplace=True)
music.reset_index(drop=True, inplace=True)
print('After dropna:', music.shape)

After dropna: (4466, 5)


## 3. Feature Engineering — Build Tags

In [8]:
def collapse(text):
    return str(text).replace(' ', '')

music['track_artist'] = music['track_artist'].apply(collapse)
music['playlist_genre'] = music['playlist_genre'].apply(collapse)
music['playlist_subgenre'] = music['playlist_subgenre'].apply(collapse)

In [9]:
music['tags'] = (
    music['track_artist'] + ' ' +
    music['playlist_genre'] + ' ' +
    music['playlist_subgenre']
)
music['tags'] = music['tags'].apply(lambda x: x.lower())
music[['track_name', 'track_artist', 'tags']].head()

,track_name,track_artist,tags
0,Die With A Smile,"LadyGaga,BrunoMars","ladygaga,brunomars pop mainstream"
1,BIRDS OF A FEATHER,BillieEilish,billieeilish pop mainstream
2,That’s So True,GracieAbrams,gracieabrams pop mainstream
3,Taste,SabrinaCarpenter,sabrinacarpenter pop mainstream
4,APT.,"ROSÉ,BrunoMars","rosé,brunomars pop mainstream"


## 4. Build Final DataFrame

In [10]:
new_df = music[['track_id', 'track_name', 'track_artist', 'tags']].copy()
new_df.rename(columns={'track_name': 'title', 'track_id': 'song_id'}, inplace=True)
new_df.reset_index(drop=True, inplace=True)
new_df.head()

,song_id,title,track_artist,tags
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"LadyGaga,BrunoMars","ladygaga,brunomars pop mainstream"
1,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,BillieEilish,billieeilish pop mainstream
2,7ne4VBA60CxGM75vw0EYad,That’s So True,GracieAbrams,gracieabrams pop mainstream
3,1d7Ptw3qYcfpdLNL5REhtJ,Taste,SabrinaCarpenter,sabrinacarpenter pop mainstream
4,5vNRhkKd0yEAg8suGBpjeY,APT.,"ROSÉ,BrunoMars","rosé,brunomars pop mainstream"


## 5. Vectorization

In [11]:
# Convert tags into vectors — same as movie recommender
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
print('Vectors shape:', vectors.shape)

Vectors shape: (4466, 4216)


In [12]:
cv.get_feature_names_out()[:20]

array(['0contactos', '112', '13organisé', '182', '1994', '1dabanton',
       '21savage', '24kgoldn', '2baba', '2jtherichest', '2pac', '2wei',
       '2wobunnies', '2woshort', '311', '38special', '3blokes',
       '3doorsdown', '40', '41'], dtype=object)

## 6. Cosine Similarity

In [13]:
similarity = cosine_similarity(vectors)
print('Similarity matrix shape:', similarity.shape)

Similarity matrix shape: (4466, 4466)


## 7. Recommendation Function

In [14]:
def recommend(song):
    matches = new_df[new_df['title'] == song]
    if matches.empty:
        print(f"Song '{song}' not found!")
        return
    
    song_index = matches.index[0]
    distances = similarity[song_index]
    
    songs_sorted = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"Songs similar to '{song}':")
    print('-' * 40)
    for i in songs_sorted:
        song_data = new_df.iloc[i[0]]
        print(f"🎵 {song_data['title']} — {song_data['track_artist']}")

In [15]:
recommend(new_df['title'][0])

Songs similar to 'Die With A Smile':
----------------------------------------
🎵 Disease — LadyGaga
🎵 APT. — ROSÉ,BrunoMars
🎵 BIRDS OF A FEATHER — BillieEilish
🎵 That’s So True — GracieAbrams
🎵 Taste — SabrinaCarpenter


In [16]:
recommend(new_df['title'][5])

Songs similar to 'Good Luck, Babe!':
----------------------------------------
🎵 HOT TO GO! — ChappellRoan
🎵 BIRDS OF A FEATHER — BillieEilish
🎵 That’s So True — GracieAbrams
🎵 Taste — SabrinaCarpenter
🎵 Diet Pepsi — AddisonRae


In [18]:
new_df['title'].values

array(['Die With A Smile', 'BIRDS OF A FEATHER', 'That’s So True', ...,
       'Sarasuda Varnam (Raga Saveri, Aadi Tala): Jaya Jaya (Raga Nattai, Khanda Chapu Tala)',
       'Raga Karnaranjani', 'Nagumomu - Abheri - Adi - Live'],
      shape=(4466,), dtype=object)

## 8. Save Pickle Files

In [19]:
pickle.dump(new_df, open('songs.pkl', 'wb'))

similarity = similarity.astype(np.float32)
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print('songs.pkl and similarity.pkl saved!')

songs.pkl and similarity.pkl saved!
